# Cleaning & Metadata Enrichment & Chunking

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Clean markdown (light cleaning)

In [ ]:
import os
import re
from pathlib import Path
from tqdm import tqdm

In [ ]:
# CONFIGURATION
# ==============================

BASE_DIR = Path("/content/drive/MyDrive")  # change if needed

PAPERS_DIR = BASE_DIR / "Papers Processing" / "extracted_markdown"
BOOKS_DIR = BASE_DIR / "Books Processing" / "extracted_markdown"

OUTPUT_PAPERS = BASE_DIR / "Papers Processing" / "cleaned"
OUTPUT_BOOKS = BASE_DIR / "Books Processing" / "cleaned"

OUTPUT_PAPERS.mkdir(parents=True, exist_ok=True)
OUTPUT_BOOKS.mkdir(parents=True, exist_ok=True)

### Cleaning Functions

In [ ]:
# =========================
# UNICODE FIX
# =========================
def fix_unicode(text):
    replacements = {
        "ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff",
        "ﬃ": "ffi", "ﬄ": "ffl",
        "’": "'", "“": '"', "”": '"',
        "–": "-", "—": "-"
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text


# =========================
# SAFE TABLE REMOVAL (FIXED)
# =========================
def remove_tables(text):
    # only remove real markdown tables (not random | usage)
    lines = text.split("\n")
    cleaned = []
    skip_block = False

    for line in lines:
        if re.match(r'^\s*\|.*\|\s*$', line):
            skip_block = True
            continue
        if skip_block and re.match(r'^\s*\|?[-: ]+\|[-: |]*$', line):
            continue
        else:
            skip_block = False
            cleaned.append(line)

    return "\n".join(cleaned)

# =========================
# REMOVE EMAILS / URLS (SAFE)
# =========================
def remove_contacts(text):
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)
    text = re.sub(r'http\S+|www\.\S+', '[URL]', text)
    return text


# =========================
# REMOVE IMAGE MARKDOWN
# =========================
def remove_images(text):
    return re.sub(r'!\[.*?\]\(.*?\)', '[Image]', text)


# =========================
# FIGURES / TABLE LABELS
# =========================
def remove_figures(text):
    text = re.sub(r'Figure\s*\d+.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Table\s*\d+.*', '', text, flags=re.IGNORECASE)
    return text


# =========================
# REMOVE REFERENCES (SAFE CUT)
# =========================
def remove_references(text):
    match = re.search(r'(?im)^\s*(references|bibliography)\s*$', text)
    if match:
        return text[:match.start()]
    return text


# =========================
# LIGHT HEADER FIX
# =========================
def fix_headers(text):
    # "#AbstractSomething" → "# Abstract\nSomething"
    text = re.sub(r'(#\s*[A-Za-z]+)([A-Z])', r'\1\n\2', text)

    # Ensure space after #
    text = re.sub(r'#([A-Za-z])', r'# \1', text)

    return text


# =========================
# 6. REMOVE PAGE NUMBERS (SAFE)
# =========================
def remove_page_numbers(text):
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*Page\s*\d+.*$', '', text, flags=re.MULTILINE)
    return text


# =========================
# FIX SPACING (SAFE)
# =========================
def fix_spacing(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text


# =========================
# 7. REMOVE HEADER/FOOTER (SMART)
# =========================
def remove_repeated_headers(text):
    lines = text.split("\n")
    freq = {}

    for line in lines:
        l = line.strip()
        if 10 < len(l) < 100:
            freq[l] = freq.get(l, 0) + 1

    cleaned = [
        line for line in lines
        if freq.get(line.strip(), 0) < 8  # only remove VERY repeated
    ]

    return "\n".join(cleaned)


# =========================
# REMOVE BOOK COPYRIGHT (SAFE)
# =========================
def remove_book_noise(text):
    patterns = [
        r'All rights reserved.*?\n',
        r'ISBN:.*?\n',
        r'Printed in.*?\n',
        r'No part of this publication.*',
        r'Copyright.*'
    ]

    for p in patterns:
        text = re.sub(p, '', text, flags=re.IGNORECASE)

    return text


# =========================
# SAFE NOISE FILTER (IMPORTANT FIX)
# =========================
def remove_noise_lines(text):
    lines = text.split("\n")
    cleaned = []

    for line in lines:
        s = line.strip()

        # DO NOT remove short lines blindly
        if len(s) == 0:
            continue

        # remove only pure garbage lines (safe)
        if re.fullmatch(r'[\W_]{5,}', s):  # only long symbol-only lines
            continue

        cleaned.append(line)

    return "\n".join(cleaned)


# =========================
# 9. NORMALIZE WHITESPACE
# =========================
def normalize_whitespace(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

In [ ]:
# MASTER CLEAN FUNCTION
# =========================
def clean_markdown_safe(text, is_book=False):

    text = fix_unicode(text)

    text = remove_tables(text)
    text = remove_images(text)
    text = remove_contacts(text)
    text = remove_figures(text)

    text = remove_page_numbers(text)
    text = remove_repeated_headers(text)

    if is_book:
        text = remove_book_noise(text)

    text = fix_headers(text)
    text = remove_references(text)
    text = fix_spacing(text)

    text = remove_noise_lines(text)
    text = normalize_whitespace(text)

    return text

In [ ]:
def process_dataset(input_dir, output_dir, dataset_name="papers"):

    print(f"\n🔄 Processing {dataset_name}...\n")

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    folders = [f for f in input_dir.iterdir() if f.is_dir()]

    success, failed = 0, 0

    for folder in tqdm(folders):

        try:
            md_files = list(folder.glob("*.md"))
            if not md_files:
                continue

            md_file = md_files[0]

            text = md_file.read_text(encoding="utf-8", errors="ignore")

            is_book = (dataset_name.lower() == "books")

            cleaned_text = clean_markdown_safe(text, is_book=is_book)

            doc_id = str(folder.name)
            output_file = output_dir / f"{doc_id}.md"

            output_file.write_text(cleaned_text, encoding="utf-8")

            success += 1

        except Exception as e:
            print(f"❌ Error in {folder.name}: {e}")
            failed += 1

    print(f"\n✅ Done {dataset_name}")
    print(f"✔ Success: {success}")
    print(f"❌ Failed: {failed}")

In [ ]:
# RUN
# ==============================

print("🚀 Starting Cleaning Pipeline...\n")

process_dataset(PAPERS_DIR, OUTPUT_PAPERS, "Papers")
process_dataset(BOOKS_DIR, OUTPUT_BOOKS, "Books")

print("\n✅ Cleaning Completed Successfully!")

🚀 Starting Cleaning Pipeline...


🔄 Processing Papers...



100%|██████████| 360/360 [05:11<00:00,  1.15it/s]



✅ Done Papers
✔ Success: 360
❌ Failed: 0

🔄 Processing Books...



100%|██████████| 1/1 [00:00<00:00,  6.87it/s]


✅ Done Books
✔ Success: 1
❌ Failed: 0

✅ Cleaning Completed Successfully!


## Build Metadata Index

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [ ]:
# CONFIG
# =========================
BASE_DIR = Path("/content/drive/MyDrive")

PAPERS_META = BASE_DIR / "Papers Processing" / "extracted_markdown"
BOOKS_META  = BASE_DIR / "Books Processing" / "extracted_markdown"

OUTPUT_FILE = BASE_DIR / "Data" / "final" / "metadata_index.csv"
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# TITLE EXTRACTOR
# =========================
def extract_title(meta_json):
    """
    Extract title from computed_toc
    """
    try:
        if "computed_toc" in meta_json and len(meta_json["computed_toc"]) > 0:
            title = meta_json["computed_toc"][0].get("title", None)
            if title:
                return title.replace("\n", " ").strip()
    except:
        pass

    return "Unknown Title"

In [ ]:
# PROCESS ONE CATEGORY
# =========================
def process_folder(base_folder, doc_type):
    rows = []

    # each folder = one document (id)
    for doc_id in tqdm(os.listdir(base_folder)):
        doc_id = str(doc_id)
        doc_path = base_folder / doc_id

        if not doc_path.is_dir():
            continue

        # find meta json
        meta_file = None
        for f in os.listdir(doc_path):
            if f.endswith("_meta.json"):
                meta_file = doc_path / f
                break

        title = "Unknown Title"

        # read metadata if exists
        if meta_file and meta_file.exists():
            try:
                with open(meta_file, "r", encoding="utf-8") as f:
                    meta_json = json.load(f)
                    title = extract_title(meta_json)
            except:
                pass

        # SPECIAL CASE (your single missing metadata paper)
        if doc_id == "2207.04672":
            title = "No Language Left Behind: Scaling Human-Centered Machine Translation"

        # file path (IMPORTANT FORMAT YOU REQUESTED)
        file_path = f"Data/cleaned/{'papers' if doc_type=='paper' else 'books'}/{doc_id}.md"

        rows.append({
            "id": doc_id,
            "title": title,
            "type": doc_type,
            "file_path": file_path
        })

    return rows

In [ ]:
# RUN PIPELINE
# =========================
all_rows = []

print("Processing Papers...")
all_rows += process_folder(PAPERS_META, "paper")

print("Processing Books...")
all_rows += process_folder(BOOKS_META, "book")

# convert to dataframe
df = pd.DataFrame(all_rows)

# save
df.to_csv(OUTPUT_FILE, index=False)

print("\n✅ Metadata index created successfully!")
print(f"Saved at: {OUTPUT_FILE}")
print(f"Total records: {len(df)}")

Processing Papers...


100%|██████████| 360/360 [04:45<00:00,  1.26it/s]


Processing Books...


100%|██████████| 1/1 [00:00<00:00,  1.18it/s]



✅ Metadata index created successfully!
Saved at: /content/drive/MyDrive/Data/final/metadata_index.csv
Total records: 361


## Chunking (RAG-ready dataset)

In [ ]:
import re
import json
from pathlib import Path
from tqdm import tqdm

In [ ]:
# CONFIG
# =========================
BASE_DIR = Path("/content/drive/MyDrive")

CLEANED_PAPERS = BASE_DIR / "Papers Processing" / "cleaned"
CLEANED_BOOKS  = BASE_DIR / "Books Processing" / "cleaned"

OUTPUT_FILE = BASE_DIR / "Data" / "processed" / "chunks.jsonl"
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 800     # words
OVERLAP = 80        # words

In [ ]:
# SPLIT INTO SECTIONS (SMART)
# =========================
def split_sections(text):
    pattern = r'(?m)^#{1,3}\s+(.+)'
    matches = list(re.finditer(pattern, text))

    if not matches:
        return [("full_text", text)]

    sections = []

    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)

        title = matches[i].group(1).strip()
        content = text[start:end].strip()

        sections.append((title, content))

    return sections

def remove_headers_from_section(text):
    lines = text.split("\n")
    return "\n".join([l for l in lines if not l.startswith("#")])


def is_bad_chunk(chunk):
    if len(chunk.split()) < 50:
        return True
    if chunk.count("|") > 5:
        return True
    return False

In [ ]:
# WORD CHUNKING WITH OVERLAP
# =========================
def chunk_text(text, size=400, overlap=80):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        start += (size - overlap)

    return chunks

In [ ]:
# PROCESS ONE DOCUMENT
# =========================
def process_document(file_path, doc_type):

    doc_id = file_path.stem
    text = file_path.read_text(encoding="utf-8", errors="ignore")

    sections = split_sections(text)

    results = []
    chunk_counter = 0

    for sec_idx, (section_title, section_text) in enumerate(sections):

        parent_text = section_text.strip()

        child_chunks = chunk_text(section_text)

        for i, chunk in enumerate(child_chunks):
            chunk_counter += 1

            results.append({
                "chunk_id": f"{doc_id}_sec{sec_idx}_chunk_{i:03d}",
                "doc_id": doc_id,
                "type": doc_type,
                "section": section_title,
                "hierarchy": section_title,
                "parent_text": parent_text,
                "text": chunk,
                "tokens": len(chunk.split())
            })

    return results

In [ ]:
# RUN PIPELINE
# =========================
all_chunks = []

print("📄 Processing Papers...")
for file in tqdm(list(CLEANED_PAPERS.glob("*.md"))):
    all_chunks.extend(process_document(file, "paper"))

print("📚 Processing Books...")
for file in tqdm(list(CLEANED_BOOKS.glob("*.md"))):
    all_chunks.extend(process_document(file, "book"))

# save JSONL
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for row in all_chunks:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("\n✅ Ultra Chunking Completed!")
print(f"📦 Total chunks: {len(all_chunks)}")
print(f"📁 Saved to: {OUTPUT_FILE}")

📄 Processing Papers...


100%|██████████| 360/360 [00:02<00:00, 129.27it/s]


📚 Processing Books...


100%|██████████| 1/1 [00:00<00:00, 38.33it/s]



✅ Ultra Chunking Completed!
📦 Total chunks: 17590
📁 Saved to: /content/drive/MyDrive/Data/processed/chunks.jsonl


Phase 1:
- Clean markdown (light cleaning)
- Build metadata index (CSV)

Phase 2:
- Split by headers
- Then chunk by tokens (800–1000)

Phase 3:
- Create chunks.jsonl
- Add metadata fields

Phase 4:
- Build embeddings (FAISS)
- RAG retrieval